In [1]:
%pip install sqlalchemy pymysql


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install cryptography


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [4]:
load_dotenv()

username = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

connection_string = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
engine = create_engine(connection_string)

print("Connection variables loaded.")

Connection variables loaded.


In [6]:
test_query = "SELECT * FROM rental LIMIT 5;"
test_df = pd.read_sql(test_query, engine)
display(test_df)

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [7]:
def rentals_month(engine, month, year):
    query = f"""
    SELECT *
    FROM rental
    WHERE MONTH(rental_date) = {month}
      AND YEAR(rental_date) = {year}
    """
    
    df = pd.read_sql(query, engine)
    return df

In [8]:
may_rentals = rentals_month(engine, 5, 2005)
june_rentals = rentals_month(engine, 6, 2005)

display(may_rentals.head())
display(june_rentals.head())

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1158,2005-06-14 22:53:33,1632,416,2005-06-18 21:37:33,2,2006-02-15 21:30:53
1,1159,2005-06-14 22:55:13,4395,516,2005-06-17 02:11:13,1,2006-02-15 21:30:53
2,1160,2005-06-14 23:00:34,2795,239,2005-06-18 01:58:34,2,2006-02-15 21:30:53
3,1161,2005-06-14 23:07:08,1690,285,2005-06-21 17:12:08,1,2006-02-15 21:30:53
4,1162,2005-06-14 23:09:38,987,310,2005-06-23 22:00:38,1,2006-02-15 21:30:53


In [9]:
def rental_count_month(df, month, year):
    column_name = f"rentals_{month:02d}_{year}"
    
    result = (
        df.groupby("customer_id")
          .size()
          .reset_index(name=column_name)
    )
    
    return result

In [10]:
may_counts = rental_count_month(may_rentals, 5, 2005)
june_counts = rental_count_month(june_rentals, 6, 2005)

display(may_counts.head())
display(june_counts.head())

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


,customer_id,rentals_06_2005
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5


In [11]:
def compare_rentals(df1, df2):
    merged = pd.merge(df1, df2, on="customer_id", how="inner")
    
    rental_cols = [col for col in merged.columns if col != "customer_id"]
    merged["difference"] = merged[rental_cols[1]] - merged[rental_cols[0]]
    
    return merged

In [12]:
comparison = compare_rentals(may_counts, june_counts)
display(comparison.head())

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,5,3,5,2
4,6,3,4,1


In [13]:
comparison_sorted = comparison.sort_values("difference", ascending=False)
display(comparison_sorted.head(10))

,customer_id,rentals_05_2005,rentals_06_2005,difference
386,454,1,10,9
389,457,1,9,8
178,213,1,9,8
248,295,1,9,8
23,27,1,8,7
194,234,1,8,7
218,260,1,8,7
322,380,1,8,7
481,561,2,9,7
345,406,1,7,6


## Conclusion

- I connected Python to the Sakila database and retrieved rental data for May and June 2005.

- I then counted the number of rentals per customer for each month and compared the results for customers who were active in both months.

- The comparison shows that many active customers rented more films in June than in May.

In [14]:
comparison_sorted.head(10)

,customer_id,rentals_05_2005,rentals_06_2005,difference
386,454,1,10,9
389,457,1,9,8
178,213,1,9,8
248,295,1,9,8
23,27,1,8,7
194,234,1,8,7
218,260,1,8,7
322,380,1,8,7
481,561,2,9,7
345,406,1,7,6
